In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [4]:
import warnings

import json
from numpy import random

from model.optimization import load_model, eval_model
from model.dataset import get_dataframe
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.model_selection import train_test_split

import lightgbm as lgb

DEFAULT_RANDOM_SEED = 774
random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
warnings.filterwarnings("ignore")

category="min_tpm_5"

In [5]:
test_data = get_dataframe(category=category, dataset="test")
subtypes = test_data["subtype"].unique()

In [6]:
def run_tests(genes: int, report: bool = True):
  importances = json.loads(open(f"../../preprocessed/{category}/important_genes_random_forest_f1.json").readline())
  chosen_genes = list(set([y["gene"] for x in [sex_values[:genes] for subtype_items in importances.values() for sex_values in subtype_items.values()] for y in x]))
  print(f"Total chosen genes: {len(chosen_genes)}")

  train_data = get_dataframe(category=category, dataset="train")
  X, y = train_data[chosen_genes], train_data["subtype"]

  X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y)

  model = load_model(
    name="lgbm_feature_selection_random_forest_f1",
    model_factory=lambda **kwargs: lgb.LGBMClassifier(**kwargs, class_weight="balanced", verbosity=-1, n_jobs=-1, random_state=DEFAULT_RANDOM_SEED),
    dataset=(X_train, y_train)
  )
  
  model = CalibratedClassifierCV(FrozenEstimator(model), method="sigmoid")
  model.fit(X_valid, y_valid)

  return eval_model(model, dataset=(test_data[chosen_genes], test_data["subtype"]), report=report, use_threshold=False)
  

In [9]:
results = run_tests(genes=5)
print(results.report)

Total chosen genes: 90
   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.827742   0.79597      0.785116           0.785116  0.831395
              precision    recall  f1-score   support

     BCRABL1       0.62      0.42      0.50        12
     DUX4IGH       0.91      1.00      0.95        21
   ETV6RUNX1       1.00      0.94      0.97        33
       HYPER       0.71      0.87      0.78        23
        HYPO       0.88      0.64      0.74        11
       KMT2A       0.95      0.95      0.95        19
        PAX5       0.79      0.85      0.81        13
      PHlike       0.62      0.70      0.65        23
    TCF3PBX1       1.00      1.00      1.00        11
      iAMP21       0.75      0.50      0.60         6

    accuracy                           0.83       172
   macro avg       0.82      0.79      0.80       172
weighted avg       0.83      0.83      0.83       172

